# 05 — SD3 Medium time-adaptive SAE circuits

Этот ноутбук заменяет старый смешанный экспериментальный блок на **SD3 Medium-only** pipeline:

1. генерация 150 изображений и повторное извлечение **full SD3 joint-attention probability maps** на раннем, среднем и позднем шагах;
2. параллельное извлечение image-token residual updates для SAE training/steering;
3. CLIPSeg-псевдомаски для объектов из промптов;
4. localization diagnostics по full probability maps: slice `image queries → text keys`;
5. Top-K SAE для early/mid/late layer-step пар;
6. mask-aligned concept dictionaries;
7. causal probes: block residual scaling и active-token SAE steering;
8. таблицы, графики, diff-метрики и визуальные сетки прямо в ноутбуке.

Решение по артефактам: для задачи circuit/localization сохраняются **head-averaged full joint-attention probability maps** SD3, а для SAE steering сохраняются **image-token residual updates**. Это не взаимоисключающие артефакты: probability maps дают evidence для attention/circuit edges, residual stream нужен для sparse features и интервенций.


In [ ]:
from pathlib import Path
import os
import sys
import json
import math
import subprocess

import numpy as np
import pandas as pd
from IPython.display import display


def _looks_like_repo(path: Path) -> bool:
    return (path / "src" / "diffusion_attention_analysis_v2").exists() and (path / "configs").exists()


def _find_project_root() -> Path:
    candidates = []
    if os.environ.get("PROJECT_ROOT"):
        candidates.append(Path(os.environ["PROJECT_ROOT"]))
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.extend([
        Path("/home/jupyter/project/diffusion_attention_analysis_2"),
        Path("/home/jupyter/datasphere/project/diffusion_attention_analysis_2"),
    ])
    for candidate in candidates:
        if _looks_like_repo(candidate.expanduser().resolve()):
            return candidate.expanduser().resolve()
    raise FileNotFoundError("Cannot locate diffusion_attention_analysis_2 repository. Set PROJECT_ROOT manually.")


PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.environ["PYTHONPATH"] = str(SRC) + os.pathsep + os.environ.get("PYTHONPATH", "")

from diffusion_attention_analysis_v2.notebook_runner import run_config, show_report
from diffusion_attention_analysis_v2.sd3_notebook_utils import (
    activation_inventory,
    attention_inventory,
    collect_intervention_diffs,
    directory_size_bytes,
    estimate_sd3_storage,
    feature_specificity,
    find_project_root,
    gib,
    make_image_grid,
    prepare_datashere_storage,
    safe_file_part,
    select_features_for_labels,
    write_label_probe_prompts,
    write_sd3_circuit_prompts,
)

# Re-resolve through the packaged helper after sys.path is ready.
PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("Python =", sys.executable)
print("cwd =", Path.cwd())


## 0. Path management: container4 and cache layout

Все крупные артефакты, model cache и pip cache направляются в `container4`. В `PROJECT_ROOT/outputs/<RUN_TAG>` и `PROJECT_ROOT/annotations/<RUN_TAG>` создаются symlink-и на filestore, поэтому stage configs остаются относительными и совместимыми с CLI.


In [ ]:
RUN_TAG = "sd3_medium_sae_circuit_150"
CONTAINER_NAME = "container4"

storage_estimate = estimate_sd3_storage(prompt_count=150)
storage_info = prepare_datashere_storage(
    project_root=PROJECT_ROOT,
    run_tag=RUN_TAG,
    container_name=CONTAINER_NAME,
    require_container=True,
    min_free_gb=storage_estimate["recommended_container4_gib_rounded"],
)

non_tensor_artifacts = (
    storage_estimate["artifacts_without_model_or_pip_gib"]
    - storage_estimate["residual_stream_total_gib"]
    - storage_estimate["full_joint_probability_maps_head_averaged_total_gib"]
)

estimate_rows = [
    {"component": "Full SD3 joint-attention maps, head-averaged, 1350 files", "GiB": storage_estimate["full_joint_probability_maps_head_averaged_total_gib"]},
    {"component": "Residual activations for SAE, 1350 files", "GiB": storage_estimate["residual_stream_total_gib"]},
    {"component": "Images + masks + reports + SAE checkpoints", "GiB": non_tensor_artifacts},
    {"component": "HF/model cache budget", "GiB": storage_estimate["model_cache_gib_budget"]},
    {"component": "pip cache budget", "GiB": storage_estimate["pip_cache_gib_budget"]},
    {"component": "safety margin", "GiB": storage_estimate["safety_margin_gib"]},
    {"component": "recommended minimum", "GiB": storage_estimate["recommended_container4_gib"]},
    {"component": "rounded minimum", "GiB": storage_estimate["recommended_container4_gib_rounded"]},
    {"component": "practical DataSphere allocation", "GiB": max(240.0, float(storage_estimate["recommended_container4_gib_rounded"]))},
]

display(pd.DataFrame(estimate_rows))
display(pd.DataFrame([storage_info]))

print("Full SD3 joint probability maps:")
print("  head-averaged per file GiB:", round(storage_estimate["full_joint_probability_maps_head_averaged_per_file_gib"], 4))
print("  head-averaged total GiB:", round(storage_estimate["full_joint_probability_maps_head_averaged_total_gib"], 2))
print("  headwise total GiB, not enabled:", round(storage_estimate["full_joint_probability_maps_headwise_total_gib"], 2))
print("Chosen artifacts:", storage_estimate["primary_artifact_choice"])

if not storage_info["free_space_ok"]:
    print("WARNING: free space is below the rounded minimum. Increase container4 or reduce prompt_count/modules/steps.")


## 1. Experiment constants and run switches

The default settings run the complete SD3 Medium experiment. For debugging, set any `RUN_*` flag to `False`; dry-run plans still show what would be executed.


In [ ]:
PROMPTS_PATH = PROJECT_ROOT / "prompts" / "sd3_medium_sae_circuit_150.jsonl"
CAPTURE_DIR = PROJECT_ROOT / "outputs" / RUN_TAG / "01_capture"
DIAG_DIR = PROJECT_ROOT / "outputs" / RUN_TAG / "02_attention_localization"
MASK_DIR = PROJECT_ROOT / "annotations" / RUN_TAG
SAE_ROOT = PROJECT_ROOT / "outputs" / RUN_TAG / "03_sae"
CONCEPT_ROOT = PROJECT_ROOT / "outputs" / RUN_TAG / "04_concept_dictionary"
INTERVENTION_ROOT = PROJECT_ROOT / "outputs" / RUN_TAG / "05_interventions"
PROBE_PROMPT_ROOT = PROJECT_ROOT / "prompts" / f"{RUN_TAG}_probes"

CAPTURE_STEPS = [0, 13, 27]
TRANSFORMER_MODULES = ["transformer_blocks.0", "transformer_blocks.11", "transformer_blocks.23"]
ATTENTION_LAYERS = ["transformer_blocks.0.attn.processor", "transformer_blocks.11.attn.processor", "transformer_blocks.23.attn.processor"]

SAE_SPECS = {
    "early_block0_step0": {"layer": "transformer_blocks.0", "step": 0, "window": [0, 1, 2, 3, 4]},
    "mid_block11_step13": {"layer": "transformer_blocks.11", "step": 13, "window": [11, 12, 13, 14, 15]},
    "late_block23_step27": {"layer": "transformer_blocks.23", "step": 27, "window": [23, 24, 25, 26, 27]},
}

TARGET_LABELS = ["dog", "cat", "car", "book", "bicycle", "person"]
PROMPTS_PER_LABEL = 2

DRY_RUN_FIRST = True
RUN_CAPTURE = True
RUN_MASKS = True
RUN_DIAGNOSTICS = True
RUN_SAE_TRAINING = True
RUN_CONCEPT_DICTIONARY = True
RUN_BLOCK_SCALING = True
RUN_SAE_STEERING = True

print("RUN_TAG:", RUN_TAG)
print("Expected activation files:", 150 * len(CAPTURE_STEPS) * len(TRANSFORMER_MODULES))
print("PROMPTS_PATH:", PROMPTS_PATH)
print("CAPTURE_DIR:", CAPTURE_DIR)


## 2. Generate deterministic SD3 prompt set

The prompt file has 150 prompts, 30 object labels, and two object entities per prompt. Each label appears approximately evenly across the dataset.


In [ ]:
write_sd3_circuit_prompts(PROMPTS_PATH, n=150, force=True)
rows = [json.loads(line) for line in PROMPTS_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
label_counts = {}
for row in rows:
    for entity in row.get("entities", []):
        label_counts[entity["label"]] = label_counts.get(entity["label"], 0) + 1

print("prompts:", len(rows))
display(pd.DataFrame(sorted(label_counts.items()), columns=["label", "count"]).head(30))
display(pd.DataFrame(rows).head(5))


## 3. Stage 1–2: capture, masks, probability-map localization

Capture uses `track_step_on_denoiser_pre_hook=True`, `strict_activation_count=True` and `strict_attention_count=True`, so the previous `1347 vs 1350` issue becomes visible immediately instead of silently propagating. The expected counts are 1350 residual activation files and 1350 full joint-attention probability-map files.


In [ ]:
CAPTURE_CFG = "configs/01_capture/sd3_medium_sae_circuit_150.yaml"
MASK_CFG = "configs/02_attention_localization/sd3_medium_sae_circuit_masks_clipseg.yaml"
DIAG_CFG = "configs/02_attention_localization/sd3_medium_sae_circuit_diagnostics.yaml"

if DRY_RUN_FIRST:
    run_config(CAPTURE_CFG, dry_run=True)
if RUN_CAPTURE:
    capture_report = run_config(CAPTURE_CFG, dry_run=False)
else:
    capture_report = show_report(str(CAPTURE_DIR / "report.json"))

if DRY_RUN_FIRST:
    run_config(MASK_CFG, dry_run=True)
if RUN_MASKS:
    mask_report = run_config(MASK_CFG, dry_run=False)
else:
    mask_report = show_report(str(MASK_DIR / "report.json"))

if DRY_RUN_FIRST:
    run_config(DIAG_CFG, dry_run=True)
if RUN_DIAGNOSTICS:
    diag_report = run_config(DIAG_CFG, dry_run=False)
else:
    diag_report = show_report(str(DIAG_DIR / "report.json"))


## 4. Artifact count audit

For 150 prompts, 3 steps and 3 selected blocks/layers the expected count is 1350 activation files and 1350 attention probability-map files. This cell verifies the actual files on disk and shows missing combinations if any.


In [ ]:
act_inv = activation_inventory(CAPTURE_DIR / "captures", TRANSFORMER_MODULES, CAPTURE_STEPS)
attn_inv = attention_inventory(CAPTURE_DIR / "captures", ATTENTION_LAYERS, CAPTURE_STEPS)

print("activation samples:", act_inv["samples"])
print("activation expected:", act_inv["expected"], "actual:", act_inv["actual"], "missing:", len(act_inv["missing"]))
if act_inv["missing"]:
    display(pd.DataFrame(act_inv["missing"]).head(20))
assert act_inv["actual"] == act_inv["expected"], f"Activation count mismatch: {act_inv['actual']} != {act_inv['expected']}"

print("attention samples:", attn_inv["samples"])
print("attention expected:", attn_inv["expected"], "actual:", attn_inv["actual"], "missing:", len(attn_inv["missing"]))
if attn_inv["missing"]:
    display(pd.DataFrame(attn_inv["missing"]).head(20))
assert attn_inv["actual"] == attn_inv["expected"], f"Attention count mismatch: {attn_inv['actual']} != {attn_inv['expected']}"

summary_path = DIAG_DIR / "sd3_joint_attention_localization_by_layer_step.csv"
if summary_path.exists():
    loc_df = pd.read_csv(summary_path)
    display(loc_df)


## 5. Stage 3: train time-adaptive SAEs

We train three separate Top-K SAEs, one per time/layer role:

- early layout: `transformer_blocks.0`, step 0;
- mid semantic/object: `transformer_blocks.11`, step 13;
- late texture/detail: `transformer_blocks.23`, step 27.


In [ ]:
SAE_CFG = "configs/03_sae_training/sd3_medium_sae_circuit_base.yaml"
sae_reports = []
for name, spec in SAE_SPECS.items():
    out_dir = SAE_ROOT / name
    overrides = [
        f"sae.layer={spec['layer']}",
        f"sae.step={spec['step']}",
        f"data.output_dir={out_dir}",
        "sae.max_activation_files=150",
        "sae.max_tokens_per_file=2048",
    ]
    print(f"\n=== SAE {name}: {spec['layer']} step {spec['step']} ===")
    if DRY_RUN_FIRST:
        run_config(SAE_CFG, overrides=overrides, dry_run=True)
    if RUN_SAE_TRAINING:
        report = run_config(SAE_CFG, overrides=overrides, dry_run=False)
    else:
        report = show_report(str(out_dir / "report.json")) or {}
    sae_reports.append({"sae_name": name, **spec, **(report or {})})

display(pd.DataFrame([{k: v for k, v in r.items() if k not in {"history", "metadata"}} for r in sae_reports]))


## 6. Stage 4: build mask-aligned concept dictionaries

Each dictionary ranks SAE features by inside-vs-outside activation contrast under CLIPSeg object masks. The feature specificity audit penalizes features reused across many labels.


In [ ]:
CONCEPT_CFG = "configs/04_concept_dictionary/sd3_medium_sae_circuit_base.yaml"
concept_reports = []
for name, spec in SAE_SPECS.items():
    sae_path = SAE_ROOT / name / "sae.pt"
    out_dir = CONCEPT_ROOT / name
    overrides = [
        f"dictionary.layer={spec['layer']}",
        f"dictionary.step={spec['step']}",
        f"data.sae_checkpoint={sae_path}",
        f"data.output_dir={out_dir}",
        "dictionary.max_activation_files=150",
    ]
    print(f"\n=== Concept dictionary {name} ===")
    if DRY_RUN_FIRST:
        run_config(CONCEPT_CFG, overrides=overrides, dry_run=True)
    if RUN_CONCEPT_DICTIONARY:
        report = run_config(CONCEPT_CFG, overrides=overrides, dry_run=False)
    else:
        report = show_report(str(out_dir / "report.json")) or {}
    concept_reports.append({"sae_name": name, **spec, **(report or {})})

display(pd.DataFrame(concept_reports))

selection = {}
for name in SAE_SPECS:
    csv_path = CONCEPT_ROOT / name / "concept_dictionary.csv"
    if csv_path.exists():
        spec_df = feature_specificity(csv_path)
        display(spec_df.sort_values("specificity_score", ascending=False).head(12))
        selection[name] = select_features_for_labels(csv_path, TARGET_LABELS, per_label=4, max_reuse=12, min_score=0.0)

print(json.dumps(selection, indent=2, ensure_ascii=False))


## 7. Stage 5a: time-window block residual scaling

This is the causal baseline: before interpreting SAE features, verify that the selected residual stream actually affects generation in the corresponding denoising window.


In [ ]:
BLOCK_CFG = "configs/05_interventions/sd3_medium_time_adaptive_block_scaling.yaml"
block_reports = []
for name, spec in SAE_SPECS.items():
    out_dir = INTERVENTION_ROOT / "block_scaling" / name
    overrides = [
        f"data.output_dir={out_dir}",
        "data.max_prompts=4",
        f"intervention.modules=[{spec['layer']}]",
        f"intervention.target_steps={spec['window']}",
        "intervention.scales=[0.60, 0.75, 1.15, 1.35]",
    ]
    print(f"\n=== Block scaling {name}: {spec['layer']} steps={spec['window']} ===")
    if DRY_RUN_FIRST:
        run_config(BLOCK_CFG, overrides=overrides, dry_run=True)
    if RUN_BLOCK_SCALING:
        report = run_config(BLOCK_CFG, overrides=overrides, dry_run=False)
    else:
        report = show_report(str(out_dir / "report.json")) or {}
    block_reports.append({"sae_name": name, "out_dir": str(out_dir), **spec, **(report or {})})

display(pd.DataFrame(block_reports))


## 8. Stage 5b: active-token SAE steering

For each time/layer SAE and each target label, we select specificity-aware concept IDs and run active-token steering. Active-token mode modifies only the spatial image tokens where selected features already activate.


In [ ]:
SAE_STEER_CFG = "configs/05_interventions/sd3_medium_time_adaptive_sae_steering.yaml"
probe_paths = write_label_probe_prompts(PROMPTS_PATH, PROBE_PROMPT_ROOT, TARGET_LABELS, prompts_per_label=PROMPTS_PER_LABEL)
print(json.dumps(probe_paths, indent=2, ensure_ascii=False))

sae_steer_reports = []
for name, spec in SAE_SPECS.items():
    sae_path = SAE_ROOT / name / "sae.pt"
    label_to_ids = selection.get(name, {})
    for label in TARGET_LABELS:
        concept_ids = label_to_ids.get(label, [])
        prompt_path = probe_paths.get(label)
        if not concept_ids or not prompt_path:
            print(f"[skip] {name}/{label}: concept_ids={concept_ids}, prompt_path={prompt_path}")
            continue
        out_dir = INTERVENTION_ROOT / "sae_steering" / name / safe_file_part(label)
        overrides = [
            f"data.prompts_path={prompt_path}",
            f"data.sae_checkpoint={sae_path}",
            f"data.output_dir={out_dir}",
            f"data.max_prompts={PROMPTS_PER_LABEL}",
            f"intervention.module={spec['layer']}",
            f"intervention.target_steps={spec['window']}",
            f"intervention.concept_ids={concept_ids}",
            "intervention.betas=[-40.0, 0.0, 40.0, 60.0]",
            "intervention.branch_mode=cond",
            "intervention.direction_mode=mean_decoder_direction",
            "intervention.spatial_mode=active_tokens",
            "intervention.active_quantile=0.85",
            "intervention.outside_scale=0.0",
            "intervention.normalize_direction=rms",
            "intervention.beta_scale_mode=activation_rms",
        ]
        print(f"\n=== SAE steering {name}/{label}: ids={concept_ids} ===")
        if DRY_RUN_FIRST:
            run_config(SAE_STEER_CFG, overrides=overrides, dry_run=True)
        if RUN_SAE_STEERING:
            report = run_config(SAE_STEER_CFG, overrides=overrides, dry_run=False)
        else:
            report = show_report(str(out_dir / "report.json")) or {}
        sae_steer_reports.append({"sae_name": name, "label": label, "concept_ids": concept_ids, "out_dir": str(out_dir), **spec, **(report or {})})

display(pd.DataFrame(sae_steer_reports))


## 9. Collect diff metrics

Metrics compare each same-seed intervention image with the baseline image from Stage 1. They are not a semantic success metric; they quantify whether a causal intervention changed the image and how strongly.


In [ ]:
diff_tables = []
baseline_samples = CAPTURE_DIR / "samples"

for name, spec in SAE_SPECS.items():
    stage_dir = INTERVENTION_ROOT / "block_scaling" / name
    diff_tables.append(collect_intervention_diffs(
        stage_dir,
        baseline_samples,
        image_name="ablation.png",
        kind="block_scaling",
        extra={"sae_name": name, "window": name},
    ))

for name in SAE_SPECS:
    for label in TARGET_LABELS:
        stage_dir = INTERVENTION_ROOT / "sae_steering" / name / safe_file_part(label)
        diff_tables.append(collect_intervention_diffs(
            stage_dir,
            baseline_samples,
            image_name="steered.png",
            kind="sae_steering",
            extra={"sae_name": name, "window": name, "label": label},
        ))

diff_df = pd.concat([d for d in diff_tables if d is not None and not d.empty], ignore_index=True) if any(d is not None and not d.empty for d in diff_tables) else pd.DataFrame()
if not diff_df.empty:
    diff_df["artifact_flag"] = (diff_df["l1_mean"] > 0.35) | (diff_df["changed_px_020"] > 0.70)
    diff_df["moderate_change"] = diff_df["l1_mean"].between(0.03, 0.30) & (~diff_df["artifact_flag"])
    out_csv = INTERVENTION_ROOT / "sd3_time_adaptive_diff_metrics.csv"
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    diff_df.to_csv(out_csv, index=False)
    display(diff_df.head(20))
    summary = diff_df.groupby(["kind", "window"], as_index=False).agg(
        n=("sample_id", "count"),
        mean_l1=("l1_mean", "mean"),
        median_l1=("l1_mean", "median"),
        changed_px_020=("changed_px_020", "mean"),
        artifact_rate=("artifact_flag", "mean"),
        moderate_rate=("moderate_change", "mean"),
    )
    display(summary)
else:
    print("No diff metrics collected yet. Run interventions first.")


## 10. Visualizations


In [ ]:
import matplotlib.pyplot as plt

if not diff_df.empty:
    summary_plot = diff_df.groupby(["kind", "window"], as_index=False)["l1_mean"].mean()
    ax = summary_plot.pivot(index="window", columns="kind", values="l1_mean").plot(kind="bar", figsize=(9, 4))
    ax.set_title("Mean L1 image change by intervention kind and time window")
    ax.set_ylabel("mean L1")
    ax.set_xlabel("time/layer window")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.show()

    top = diff_df.sort_values("l1_mean", ascending=False).head(12)
    display(top[["kind", "window", "label", "beta", "scale", "l1_mean", "changed_px_020", "artifact_flag", "prompt"]])
else:
    print("Run Stage 5 first to draw plots.")


In [ ]:
from IPython.display import display as ipy_display

if not diff_df.empty:
    top = diff_df.sort_values("l1_mean", ascending=False).head(8)
    image_paths = []
    titles = []
    for _, row in top.iterrows():
        image_paths.extend([row["baseline_path"], row["edited_path"]])
        titles.extend(["baseline", f"{row['kind']} {row.get('label', '')} L1={row['l1_mean']:.3f}"])
    grid = make_image_grid(image_paths, titles, ncols=4, thumb=(220, 220))
    if grid is not None:
        ipy_display(grid)
else:
    print("No images to show yet.")


## 11. Artifact inventory and final storage check


In [ ]:
paths_for_size = [
    ("outputs/run", PROJECT_ROOT / "outputs" / RUN_TAG),
    ("annotations/run", PROJECT_ROOT / "annotations" / RUN_TAG),
    ("hf_home", Path(os.environ["HF_HOME"])),
    ("pip_cache", Path(os.environ["PIP_CACHE_DIR"])),
]
size_rows = []
for label, path in paths_for_size:
    size_rows.append({"component": label, "path": str(path), "GiB": gib(directory_size_bytes(path))})
size_df = pd.DataFrame(size_rows)
display(size_df)
print("Total GiB:", round(size_df["GiB"].sum(), 2))

reports = []
for report_path in [
    CAPTURE_DIR / "report.json",
    MASK_DIR / "report.json",
    DIAG_DIR / "report.json",
    *[SAE_ROOT / name / "report.json" for name in SAE_SPECS],
    *[CONCEPT_ROOT / name / "report.json" for name in SAE_SPECS],
]:
    if report_path.exists():
        data = json.loads(report_path.read_text(encoding="utf-8"))
        reports.append({"report": str(report_path.relative_to(PROJECT_ROOT)), **{k: data.get(k) for k in ["status", "samples", "rows", "labels", "activation_actual_files", "activation_expected_files", "activation_missing_files"]}})

display(pd.DataFrame(reports))
